In [ ]:
# ==========================================
# BIBLIOTECAS E DEPENDÊNCIAS
# ==========================================

# 1. Sistema e Matemática
import os
import math
import xml.etree.ElementTree as ET

# 2. Processamento de Imagem e Visualização
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import torchvision.transforms.functional as F
from torchvision.utils import draw_bounding_boxes

# 3. Core PyTorch e Estrutura de Dados
import torch
from torch.utils.data import Dataset, DataLoader

# 4. Ecossistema Torchvision (Transformações v2 e Operações)
import torchvision
from torchvision import tv_tensors
from torchvision.transforms import v2
from torchvision.ops import box_iou  # Essencial para avaliação clínica (IoU)

# 5. Arquitetura RetinaNet
from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.models.detection.retinanet import (
    RetinaNet_ResNet50_FPN_Weights,
    RetinaNetClassificationHead
)

# Configuração de Hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware: {device}")

In [ ]:
# ==========================================
# 1. CAMINHOS
# ==========================================
caminho_treino = r"D:\d4v\Projecto_Cotovelo\Dataset_Cotovelo_ML\Roboflow_83_Epicondyle_annotations_voc\train"
caminho_validacao = r"D:\d4v\Projecto_Cotovelo\Dataset_Cotovelo_ML\Roboflow_83_Epicondyle_annotations_voc\valid"

# ==========================================
# 2. Converter para tensor
# ==========================================
transformacoes_puras = v2.Compose([
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True) # Prepara para a placa gráfica
])

# ==========================================
# 3. CRIAR A CLASSE DO DATASET
# ==========================================
class CotoveloDataset(Dataset):
    def __init__(self, pasta_dados, transform=None):
        self.pasta_dados = pasta_dados
        self.transform = transform
        self.imagens = [img for img in os.listdir(pasta_dados) if img.endswith('.jpg')]

    def __getitem__(self, idx):
        nome_img = self.imagens[idx]
        imagem = Image.open(os.path.join(self.pasta_dados, nome_img)).convert("RGB")
        
        arvore = ET.parse(os.path.join(self.pasta_dados, nome_img.replace('.jpg', '.xml')))
        caixas = []
        for obj in arvore.getroot().findall('object'):
            xmin, ymin, xmax, ymax = [float(obj.find(f'bndbox/{coord}').text) for coord in ['xmin', 'ymin', 'xmax', 'ymax']]
            caixas.append([xmin, ymin, xmax, ymax])
            
        caixas_inteligentes = tv_tensors.BoundingBoxes(caixas, format="XYXY", canvas_size=(imagem.height, imagem.width))
        labels = torch.ones((len(caixas),), dtype=torch.int64) 
        
        if self.transform:
            imagem, caixas_inteligentes = self.transform(imagem, caixas_inteligentes)
            
        return imagem, {"boxes": caixas_inteligentes, "labels": labels}

    def __len__(self): return len(self.imagens)

def collate_fn(batch): return tuple(zip(*batch))

# Criar os DataLoaders (Treino com shuffle, Validação sem)
dataloader_treino = DataLoader(CotoveloDataset(caminho_treino, transform=transformacoes_puras), batch_size=4, shuffle=True, collate_fn=collate_fn)
dataloader_validacao = DataLoader(CotoveloDataset(caminho_validacao, transform=transformacoes_puras), batch_size=4, shuffle=False, collate_fn=collate_fn)

In [ ]:
# ==========================================
# 4. PREPARAR O MODELO E O OTIMIZADOR
# ==========================================
print("A montar a RetinaNet...")
num_classes = 2
modelo = torchvision.models.detection.retinanet_resnet50_fpn(weights=torchvision.models.detection.retinanet.RetinaNet_ResNet50_FPN_Weights.DEFAULT)
in_channels = modelo.backbone.out_channels
num_anchors = modelo.head.classification_head.num_anchors
modelo.head.classification_head = torchvision.models.detection.retinanet.RetinaNetClassificationHead(in_channels, num_anchors, num_classes)
modelo = modelo.to(device)

otimizador = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=0.0001)

# ==========================================
# 5. O CICLO DE TREINO (Train + Eval + Save)
# ==========================================
epocas = 30 # Suficiente para a tarefa de auto-cropping [cite: 104]
melhor_iou = 0.0
caminho_guardar = r"D:\d4v\Projecto_Cotovelo\models\melhor_retinanet_cotovelo.pth"

print(f"\nArrancar treino ({epocas} Épocas) na {device}!")
print("==========================================================")

for epoca in range(epocas):
    # FASE 1: TREINO
    modelo.train()
    loss_treino = 0
    for imagens, alvos in dataloader_treino:
        imagens = list(img.to(device) for img in imagens)
        alvos = [{k: v.to(device) for k, v in t.items()} for t in alvos]
        
        otimizador.zero_grad()
        dicionario_perdas = modelo(imagens, alvos)
        perda_batch = sum(loss for loss in dicionario_perdas.values())
        perda_batch.backward()
        otimizador.step()
        loss_treino += perda_batch.item()
    
    loss_media_treino = loss_treino / len(dataloader_treino)

    # FASE 2: VALIDAÇÃO
    modelo.eval()
    soma_iou = 0
    acertos = 0
    with torch.no_grad():
        for imagens, alvos in dataloader_validacao:
            imagens = list(img.to(device) for img in imagens)
            alvos = [{k: v.to(device) for k, v in t.items()} for t in alvos]
            
            previsoes = modelo(imagens)
            for i, previsao in enumerate(previsoes):
                if len(previsao['boxes']) > 0 and len(alvos[i]['boxes']) > 0:
                    # Avalia caixas com >50% de certeza
                    iou_matrix = box_iou(previsao['boxes'][previsao['scores'] > 0.5], alvos[i]['boxes'])
                    if len(iou_matrix) > 0:
                        soma_iou += iou_matrix.max(dim=1)[0].max().item()
                        acertos += 1
                        
    iou_medio_val = (soma_iou / acertos) * 100 if acertos > 0 else 0
    
    # FASE 3: CHECKPOINTING
    if iou_medio_val > melhor_iou:
        melhor_iou = iou_medio_val
        torch.save(modelo.state_dict(), caminho_guardar)
        status = f"Modelo guardado ({melhor_iou:.1f}% de IoU na Validação)!"
    else:
        status = ""
        
    print(f"Época {epoca+1:02d}/{epocas} | Loss Treino: {loss_media_treino:.4f} | IoU Validação: {iou_medio_val:.1f}% | {status}")

print("\nTreino concluido. O modelo final de produção está guardado em 'melhor_retinanet_cotovelo.pth'.")

In [ ]:
# ==============================================================================
# CÉLULA 5: AVALIAÇÃO DEFINITIVA
# ==============================================================================

# 1. CARREGAR O MELHOR MODELO GUARDADO
caminho_melhor_modelo = r"D:\d4v\Projecto_Cotovelo\models\melhor_retinanet_cotovelo.pth"

if os.path.exists(caminho_melhor_modelo):
    modelo.load_state_dict(torch.load(caminho_melhor_modelo))
    print(f"Sucesso: Pesos do melhor modelo carregados para avaliação!")
else:
    print("Aviso: Ficheiro .pth não encontrado.")

# 2. Configurar o Caminho de Teste
caminho_teste = r"D:\d4v\Projecto_Cotovelo\Dataset_Cotovelo_ML\Roboflow_83_Epicondyle_annotations_voc\test"
dataset_teste = CotoveloDataset(caminho_teste, transform=transformacoes_puras)
dataloader_teste = DataLoader(dataset_teste, batch_size=4, shuffle=False, collate_fn=collate_fn)

print(f"Dataset de Teste: {len(dataset_teste)} imagens. A iniciar avaliação...\n")
modelo.eval() 

# 3. Contadores e Métricas
verdadeiros_positivos = 0
falsos_positivos = 0
falsos_negativos = 0
soma_iou_acertos = 0.0

limiar_certeza = 0.5  
limiar_iou = 0.5      

with torch.no_grad():
    for imagens, alvos in dataloader_teste:
        imagens = list(img.to(device) for img in imagens)
        alvos = [{k: v.to(device) for k, v in t.items()} for t in alvos]
        
        previsoes = modelo(imagens)
        
        for i, previsao in enumerate(previsoes):
            caixas_reais = alvos[i]['boxes']
            indices_validos = previsao['scores'] > limiar_certeza
            caixas_preditas = previsao['boxes'][indices_validos]
            
            # Casos de ausência de deteção ou falso alarme
            if len(caixas_preditas) == 0 and len(caixas_reais) > 0:
                falsos_negativos += len(caixas_reais)
                continue
            if len(caixas_preditas) > 0 and len(caixas_reais) == 0:
                falsos_positivos += len(caixas_preditas)
                continue
            
            # Cálculo de qualidade geométrica (IoU)
            if len(caixas_preditas) > 0 and len(caixas_reais) > 0:
                iou_matrix = box_iou(caixas_preditas, caixas_reais)
                for linha_iou in iou_matrix:
                    maior_iou = linha_iou.max().item()
                    if maior_iou >= limiar_iou:
                        verdadeiros_positivos += 1
                        soma_iou_acertos += maior_iou
                    else:
                        falsos_positivos += 1
            
            if len(caixas_preditas) < len(caixas_reais):
                 falsos_negativos += (len(caixas_reais) - len(caixas_preditas))

# 4. Resultados Finais
total_objetos = verdadeiros_positivos + falsos_negativos
precisao = verdadeiros_positivos / (verdadeiros_positivos + falsos_positivos) if (verdadeiros_positivos + falsos_positivos) > 0 else 0
recall = verdadeiros_positivos / total_objetos if total_objetos > 0 else 0
iou_medio = soma_iou_acertos / verdadeiros_positivos if verdadeiros_positivos > 0 else 0

print("================================================")
print("       RELATÓRIO DE DESEMPENHO (TESTE)         ")
print("================================================")
print(f"Verdadeiros Positivos: {verdadeiros_positivos}")
print(f"Falsos Positivos:      {falsos_positivos}")
print(f"Falsos Negativos:      {falsos_negativos}")
print("------------------------------------------------")
print(f"Precisão: {precisao*100:.1f}%")
print(f"Recall:   {recall*100:.1f}%")
print(f"IoU Médio:{iou_medio*100:.1f}%")
print("================================================")

In [ ]:
# ==========================================
# CÉLULA 6: GALERIA DE TESTE
# ==========================================

# Garantir que usamos o melhor para a visualização
if os.path.exists(caminho_melhor_modelo):
    modelo.load_state_dict(torch.load(caminho_melhor_modelo))

# 1. Garantir que o modelo está em modo de avaliação
modelo.eval()

# 2. Configurar o layout da grelha
num_imagens = len(dataset_teste)
colunas = 2  
linhas = math.ceil(num_imagens / colunas)

fig, axes = plt.subplots(linhas, colunas, figsize=(15, 5 * linhas))
axes = axes.flatten()

print(f"A processar {num_imagens} imagens de teste...")

# 3. Iterar sobre todas as imagens do dataset de teste
with torch.no_grad():
    for i in range(num_imagens):
        imagem_teste, _ = dataset_teste[i]
        
        # Preparar input para o modelo
        imagem_input = imagem_teste.unsqueeze(0).to(device)
        previsao = modelo(imagem_input)[0]
        
        # Filtrar certezas (80%)
        limiar_certeza = 0.8
        indices_validos = previsao['scores'] > limiar_certeza
        caixas_preditas = previsao['boxes'][indices_validos]
        certezas = previsao['scores'][indices_validos]
        
        # Converter imagem para desenho (0-255)
        imagem_para_desenho = (imagem_teste * 255).to(torch.uint8)
        
        # Desenhar caixa se o modelo encontrou
        if len(caixas_preditas) > 0:
            etiquetas = [f"{int(c.item() * 100)}%" for c in certezas]
            imagem_final = draw_bounding_boxes(
                imagem_para_desenho, 
                boxes=caixas_preditas.cpu(), 
                labels=etiquetas, 
                colors="red", 
                width=4,
                font_size=25
            )
        else:
            imagem_final = imagem_para_desenho
            
        # Mostrar na grelha
        axes[i].imshow(F.to_pil_image(imagem_final))
        axes[i].set_title(f"Imagem {i+1}")
        axes[i].axis('off')

# Esconder subplots vazios se o número de imagens não for par
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# SCRIPT DE AUTO-CROPPING
# ==============================================================================

# 1. Configurações de Caminhos
caminho_raiz = r"D:\d4v\Projecto_Cotovelo\Dataset_Cotovelo_ML\Roboflow_83_Epicondyle_annotations_voc"
pastas_alvo = ["train", "valid", "test"]
caminho_destino = r"D:\d4v\Projecto_Cotovelo\Dataset_Recortado_Diagnostico"
caminho_pesos = r"D:\d4v\Projecto_Cotovelo\models\melhor_retinanet_cotovelo.pth"

os.makedirs(caminho_destino, exist_ok=True)

# 2. Carregar o Modelo
modelo.load_state_dict(torch.load(caminho_pesos))
modelo.eval()
modelo.to(device)

def executar_crop_clinico(caminho_img, nome_ficheiro, subpasta):
    img_pil = Image.open(caminho_img).convert("RGB")
    largura_orig, altura_orig = img_pil.size
    
    # Converter para tensor
    img_tensor = v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])(img_pil)
    
    with torch.no_grad():
        previsao = modelo(img_tensor.unsqueeze(0).to(device))[0]
    
    if len(previsao['boxes']) > 0:
        # Escolher a detecção com maior score
        idx = torch.argmax(previsao['scores']).item()
        caixa = previsao['boxes'][idx].cpu().numpy()
        
        xmin, ymin, xmax, ymax = caixa
        cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
        w_box, h_box = xmax - xmin, ymax - ymin
        
        # EXPANSÃO 5X
        nova_metade_w = (w_box * 5) / 2
        nova_metade_h = (h_box * 5) / 2
        
        crop_xmin = max(0, cx - nova_metade_w)
        crop_ymin = max(0, cy - nova_metade_h)
        crop_xmax = min(largura_orig, cx + nova_metade_w)
        crop_ymax = min(altura_orig, cy + nova_metade_h)
        
        recorte = img_pil.crop((crop_xmin, crop_ymin, crop_xmax, crop_ymax))
        
        # PADRONIZAÇÃO 800x800 COM PADDING PRETO
        recorte.thumbnail((800, 800), Image.Resampling.LANCZOS)
        delta_w = 800 - recorte.size[0]
        delta_h = 800 - recorte.size[1]
        padding = (delta_w//2, delta_h//2, delta_w-(delta_w//2), delta_h-(delta_h//2))
        imagem_final = ImageOps.expand(recorte, padding, fill="black")
        
        # Guardar com prefixo da pasta original para evitar conflitos de nomes
        nome_final = f"{subpasta}_{nome_ficheiro}"
        imagem_final.save(os.path.join(caminho_destino, nome_final))
        return True
    return False

# 3. Execução em Loop por todas as pastas
total_processadas = 0
print(f"A iniciar varredura em {caminho_raiz}...")

for sub in pastas_alvo:
    pasta_full = os.path.join(caminho_raiz, sub)
    if not os.path.exists(pasta_full): continue
    
    ficheiros = [f for f in os.listdir(pasta_full) if f.endswith('.jpg')]
    print(f"A processar {len(ficheiros)} imagens em '{sub}'...")
    
    for nome in ficheiros:
        if executar_crop_clinico(os.path.join(pasta_full, nome), nome, sub):
            total_processadas += 1

print(f"\nConcluído!")
print(f"Foram geradas {total_processadas} imagens de alta resolução (800x800) prontas para diagnóstico.")
print(f"Destino: {caminho_destino}")